# Installing MPI Library

In [1]:
%pip install mpi4py

# Creating mpi_pdc.py File ( Original )

In [2]:
%%writefile mpi_pdc.py
from mpi4py import MPI
import sys

comm = MPI.COMM_WORLD

rank = comm.Get_rank()
size = comm.Get_size()

if rank == 0:
    for i in range(1, size):
        message = comm.recv(source=i)
        print(f"Received from process {i}: {message}")
else:
  comm.send(f"Hello from process {rank}", dest=0)

Overwriting mpi_pdc.py


# For allowing the Notebook to run the next terminal prompt in root level

In [3]:
import os
os.environ["OMPI_ALLOW_RUN_AS_ROOT"] = "1"
os.environ["OMPI_ALLOW_RUN_AS_ROOT_CONFIRM"] = "1"


# Actual Terminal Prompt to Run The Code

In [10]:
!mpiexec --oversubscribe -n 4 python mpi_pdc.py

Received from process 1: Hello from process 1
Received from process 2: Hello from process 2
Received from process 3: Hello from process 3


# Added each Computation per Rank

In [11]:
%%writefile mpi_pdc.py
from mpi4py import MPI
import sys

# Initialize MPI communication world
comm = MPI.COMM_WORLD

# Get unique process ID and total number of processes
rank = comm.Get_rank()
size = comm.Get_size()

# List of operations to assign to processes
operations = ["+", "-", "/", "*"]

# Root process (coordinator)
if rank == 0:
    # Receive results from all worker processes
    for i in range(1, size):
        message = comm.recv(source=i)
        print(f"Received from process {i}: {message}")

# Worker processes
else:
    # Assign a computation task based on process rank
    assigned_task = f"{rank} {operations[rank % len(operations)]} {size}"

    # Compute result and send it to the root process
    comm.send(
        f"Hello from process {rank}! My assigned task is {assigned_task} = {eval(assigned_task)}",
        dest=0
    )

Overwriting mpi_pdc.py


In [13]:
!mpiexec --oversubscribe -n 10 python mpi_pdc.py

Received from process 1: Hello from process 1! My assigned task is 1 - 10 = -9
Received from process 2: Hello from process 2! My assigned task is 2 / 10 = 0.2
Received from process 3: Hello from process 3! My assigned task is 3 * 10 = 30
Received from process 4: Hello from process 4! My assigned task is 4 + 10 = 14
Received from process 5: Hello from process 5! My assigned task is 5 - 10 = -5
Received from process 6: Hello from process 6! My assigned task is 6 / 10 = 0.6
Received from process 7: Hello from process 7! My assigned task is 7 * 10 = 70
Received from process 8: Hello from process 8! My assigned task is 8 + 10 = 18
Received from process 9: Hello from process 9! My assigned task is 9 - 10 = -1


# Added a fail process

In [16]:
%%writefile mpi_pdc.py
from mpi4py import MPI
import sys

# Initialize MPI communication world
comm = MPI.COMM_WORLD

# Get unique process ID and total number of processes
rank = comm.Get_rank()
size = comm.Get_size()

# List of operations to assign to processes
operations = ["+", "-", "/", "*"]

# Root process (coordinator)
if rank == 0:
    # Receive results from all worker processes
    for i in range(1, size):
        message = comm.recv(source=i)
        print(f"Received from process {i}: {message}")

# Worker processes
else:
    # For failing process
    if rank == 5:
        sys.exit(1)
    # Assign a computation task based on process rank
    assigned_task = f"{rank} {operations[rank % len(operations)]} {size}"

    # Compute result and send it to the root process
    comm.send(
        f"Hello from process {rank}! My assigned task is {assigned_task} = {eval(assigned_task)}",
        dest=0
    )

Overwriting mpi_pdc.py


In [ ]:
!mpiexec --oversubscribe -n 10 python mpi_pdc.py

Received from process 1: Hello from process 1! My assigned task is 1 - 10 = -9
Received from process 2: Hello from process 2! My assigned task is 2 / 10 = 0.2
Received from process 3: Hello from process 3! My assigned task is 3 * 10 = 30
Received from process 4: Hello from process 4! My assigned task is 4 + 10 = 14


##Follow Up Questions:

###Why is message passing required in distributed systems?
- In the MPI program, each process has its own memory space, so they cannot directly access each other’s data. Message passing is required to allow processes to exchange computed results and coordinate tasks, as seen when worker processes send their results to the root process. This enables cooperation between independent processes in a distributed system.
###What happens if one process fails?
- Based on my code and output, if one process fails before sending its message, the root process waits indefinitely for that message. This causes the program to hang or terminate, showing that basic MPI programs do not handle failures automatically. This highlights the lack of built-in fault tolerance in simple message-passing models.
###How does this model differ from shared-memory programming?
- In message-passing models like MPI, processes do not share memory and must explicitly communicate using send and receive operations. In contrast, shared-memory programming allows multiple threads to access the same memory space directly. This makes MPI safer from data races but more complex due to the need for explicit communication.
